# ERCOT - RTM Price Adders Cleaning (pre-2025)
**Raw source:** `data/raw/ercot/RTM Price Adders 2021-2025/`  
**Data source:** `data/parquet/ercot/rtm_price_adders_15min_20200101_20251231.parquet`

**Outputs:** `data/02_processed/price adders/..
- **15min:** 'price_adders_15min_20200101_20251204.csv'
- **hourly:** 'price_adders_hourly_20200101_20251204.csv'
- **binary:** 'price_adders_binary_20200101_20251204.csv'

## 1. Imports

In [2]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

PROJECT_ROOT = Path('../..').resolve()
PARQUET = PROJECT_ROOT / '01_data' / '1.2_raw_api' / 'rtm_price_adders_15min_20200101_20251231.parquet'
OUT_DIR = PROJECT_ROOT / '01_data' / '2_cleaned' / 'price_adders'
OUT_DIR.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()

## 2. Schema & row count

In [3]:
schema = con.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{PARQUET}')
""").df()
print('=== Schema ===')
print(schema.to_string(index=False))

=== Schema ===
     column_name column_type null  key default extra
       timestamp   TIMESTAMP  YES None    None  None
RepeatedHourFlag     VARCHAR  YES None    None  None
        RTRSVPOR      DOUBLE  YES None    None  None
       RTRSVPOFF      DOUBLE  YES None    None  None
           RTRDP      DOUBLE  YES None    None  None
          RTRDPA      DOUBLE  YES None    None  None
         RTRDPRU      DOUBLE  YES None    None  None
         RTRDPRD      DOUBLE  YES None    None  None
        RTRDPRRS      DOUBLE  YES None    None  None
       RTRDPECRS      DOUBLE  YES None    None  None
         RTRDPNS      DOUBLE  YES None    None  None


In [4]:
summary = con.execute(f"""
    SELECT
        COUNT(*)                        AS total_rows,
        MIN(timestamp)                  AS min_ts,
        MAX(timestamp)                  AS max_ts,
        COUNT(DISTINCT timestamp)       AS unique_timestamps
    FROM read_parquet('{PARQUET}')
""").df()
print('=== Time Range ===')
print(f"Min: {summary['min_ts'].iloc[0]}, Max: {summary['max_ts'].iloc[0]}")

=== Time Range ===
Min: 2020-01-01 00:00:00, Max: 2025-12-31 23:45:00


## 3. Preview

In [5]:
# Preview
overlap = con.execute(f"""
    SELECT * FROM read_parquet('{PARQUET}')
    ORDER BY timestamp
    LIMIT 5
""").df()
print('=== Preview data before Dec 2025 ===')
overlap

=== Preview data before Dec 2025 ===


,timestamp,RepeatedHourFlag,RTRSVPOR,RTRSVPOFF,RTRDP,RTRDPA,RTRDPRU,RTRDPRD,RTRDPRRS,RTRDPECRS,RTRDPNS
0,2020-01-01 00:00:00,N,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2020-01-01 00:15:00,N,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2020-01-01 00:30:00,N,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2020-01-01 00:45:00,N,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2020-01-01 01:00:00,N,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
# export hourly PA
df_15min = con.execute(f"""
    SELECT
        timestamp, RTRSVPOR, RTRSVPOFF,	RTRDP
    FROM read_parquet('{PARQUET}')
    WHERE RepeatedHourFlag = 'N'
        AND timestamp < '2025-12-05'
    ORDER BY timestamp
""").df()

df_15min.to_csv(OUT_DIR / 'price_adders_15min_20200101_20251204.csv', index=False)
df_15min.head()

,timestamp,RTRSVPOR,RTRSVPOFF,RTRDP
0,2020-01-01 00:00:00,0.0,0.0,0.0
1,2020-01-01 00:15:00,0.0,0.0,0.0
2,2020-01-01 00:30:00,0.0,0.0,0.0
3,2020-01-01 00:45:00,0.0,0.0,0.0
4,2020-01-01 01:00:00,0.0,0.0,0.0


## 5. Aggregate and Export Hourly PA

In [26]:
# aggregate to daily average price
df_hourly = con.execute(f"""
    SELECT
        DATE_TRUNC('hour', timestamp) AS datetime,
        AVG(RTRSVPOR)  AS RTRSVPOR,
        AVG(RTRSVPOFF) AS RTRSVPOFF,
        AVG(RTRDP)     AS RTRDP
    FROM read_parquet('{PARQUET}')
    WHERE RepeatedHourFlag = 'N'
    GROUP BY DATE_TRUNC('hour', timestamp)
    ORDER BY datetime
""").df()

# filter to before Dec 2025
df_hourly = df_hourly[df_hourly['datetime'] < '2025-12-04']
df_hourly.to_csv(OUT_DIR / 'price_adders_hourly_20200101_20251204.csv', index=False)
df_hourly

,datetime,RTRSVPOR,RTRSVPOFF,RTRDP
0,2020-01-01 00:00:00,0.0,0.0,0.0
1,2020-01-01 01:00:00,0.0,0.0,0.0
2,2020-01-01 02:00:00,0.0,0.0,0.0
3,2020-01-01 03:00:00,0.0,0.0,0.0
4,2020-01-01 04:00:00,0.0,0.0,0.0
...,...,...,...,...
51925,2025-12-03 19:00:00,0.0,0.0,0.0
51926,2025-12-03 20:00:00,0.0,0.0,0.0
51927,2025-12-03 21:00:00,0.0,0.0,0.0
51928,2025-12-03 22:00:00,0.0,0.0,0.0


## 6. create binary PA for activation analysis

In [27]:
# change column values to binary, na is marked as 0
df_binary = df_hourly.copy()
for col in df_hourly.columns:
    if col != 'datetime':
        df_binary[col] = ((df_binary[col] != 0) & (df_binary[col].notna())).astype(int)

# export
df_binary.to_csv(OUT_DIR / 'price_adders_binary_20200101_20251204.csv', index=False)

df_binary

,datetime,RTRSVPOR,RTRSVPOFF,RTRDP
0,2020-01-01 00:00:00,0,0,0
1,2020-01-01 01:00:00,0,0,0
2,2020-01-01 02:00:00,0,0,0
3,2020-01-01 03:00:00,0,0,0
4,2020-01-01 04:00:00,0,0,0
...,...,...,...,...
51925,2025-12-03 19:00:00,0,0,0
51926,2025-12-03 20:00:00,0,0,0
51927,2025-12-03 21:00:00,0,0,0
51928,2025-12-03 22:00:00,0,0,0
